# WiTA v2 — Self-Contained Air-Writing Recognition (Kaggle T4)

**No `git clone` required.** All code is in this package.

| | |
|---|---|
| GPU | NVIDIA T4 (16 GB VRAM) |
| Encoder | VideoMAE-Base (MCG-NJU/videomae-base) |
| Decoder | CTC-only (attention decoder removed) |
| Target | val CER < 0.22 within 40 epochs |

**Pipeline:** install → setup → stream data → build model → train → evaluate → export

## Cell 1 — Install dependencies

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'transformers>=4.26' --quiet

import sys, os

# NOTE: this clones the public repo. After pushing local changes, re-run
# this cell (or do `cd /kaggle/working/wita_v2 && git pull`) to pick them up.
!git clone "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'


## Cell 2 — Imports & Config

In [ ]:
import os, sys, logging, random
import numpy as np
import torch


os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/logs',        exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s — %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('/kaggle/working/logs/run.log'),
    ]
)

# ── Build Config ─────────────────────────────────────────────────────
from wita_v2.configs.default import (
    Config, DataConfig, AugConfig, EncoderConfig,
    RecurrentConfig, AttnDecoderConfig, TrainConfig
)

cfg = Config(
    data=DataConfig(
        hf_repo_id  = 'yewon816/WiTA',
        lang        = 'english',
        max_zips    = None,       # ← set to 2 for a smoke test
        max_frames  = 64,
        # img_size is synced automatically from EncoderConfig.img_size via cfg.build()
    ),
    aug=AugConfig(),              # mirror_prob defaults to 0 (hflip mirrors glyphs)
    encoder=EncoderConfig(
        arch                = 'videomae',
        videomae_model_name = 'MCG-NJU/videomae-base',
        videomae_num_frames = 32,   # frames resampled to (model T input)
        tubelet_size        = 2,    # T' = 32//2 = 16 temporal tokens (fits long words)
        patch_size          = 16,
        img_size            = 224,  # VideoMAE requires 224×224
        out_dim             = 768,  # VideoMAE-B hidden size
        pretrained          = True,
        freeze_backbone     = True, # frozen forward runs under torch.no_grad → low VRAM
    ),
    recurrent=RecurrentConfig(
        arch        = 'lstm',     # 'lstm' | 'gru' | 'transformer' | 'none'
        hidden_size = 256,
        num_layers  = 2,
    ),
    attn=AttnDecoderConfig(
        n_layers = 4, n_heads = 8, ff_dim = 2048,
        # NOTE: AttnDecoderConfig kept for compat; not used in CTC-only training
    ),
    train=TrainConfig(
        num_epochs           = 100,
        batch_size           = 1,       # VideoMAE at 224px is VRAM-heavy
        accum_steps          = 8,       # effective batch = 1 × 8 = 8
        lr                   = 1e-4,
        beta2                = 0.999,   # ViT fine-tuning recipe (not 0.98)
        num_workers          = 2,
        optimizer            = 'adamw',
        scheduler            = 'onecycle',
        val_limit            = 50,      # batches per fast val pass (None = full)
        grad_clip            = 5.0,
        unfreeze_after_epoch = 10,      # unfreeze backbone after N epochs
        backbone_lr_mult     = 0.1,     # backbone LR = 0.1 × head LR (discriminative)
        checkpoint_dir       = '/kaggle/working/checkpoints',
        resume_path          = None,    # ← '/kaggle/working/checkpoints/latest.pt' to resume
        save_frequency       = 10,
        seed                 = 42,
        warmup_pct           = 0.05,
    ),
).build()   # finalises vocab indices & syncs data.img_size = encoder.img_size

# Reproducibility
random.seed(cfg.train.seed)
np.random.seed(cfg.train.seed)
torch.manual_seed(cfg.train.seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f'Device          : {cfg.device}')
print(f'Encoder         : {cfg.encoder.arch.upper()}  img_size={cfg.encoder.img_size}  '
      f"T'={cfg.encoder.T_prime}")
print(f'CTC  vocab size : {cfg.vocab.ctc_vocab_size}  (blank=0, chars 1..{len(cfg.vocab.chars)})')
print(f'Attn vocab size : {cfg.vocab.attn_vocab_size}  '
      f'(SOS={cfg.vocab.sos_idx} EOS={cfg.vocab.eos_idx} PAD={cfg.vocab.pad_idx})')
print(f'LR              : head={cfg.train.lr:.1e}  '
      f'backbone={cfg.train.lr * cfg.train.backbone_lr_mult:.1e}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## Cell 3 — Vocabulary sanity check

In [ ]:
from wita_v2.datasets.vocab import make_converter

converter = make_converter(cfg.data.lang)
print('Alphabet length:', len(converter.alphabet))

for word in ['hello', 'mississippi', 'book']:
    enc, length = converter.encode(word)
    dec = converter.decode(enc, length)
    match = '✓' if dec == word else f'✗ got "{dec}"'
    print(f'  "{word}" → {enc.tolist()} → "{dec}"  {match}')


## Cell 4 — Stream & index dataset from HuggingFace

In [ ]:
from wita_v2.datasets.dataset import stream_and_index

# Downloads ZIPs one-at-a-time, reads frames into RAM as raw bytes,
# deletes each ZIP immediately — never fills the disk with PNG files.
samples = stream_and_index(cfg)
print(f'Total samples indexed: {len(samples)}')

# Quick peek at one sample
frame_bytes, label = samples[0]
print(f'Sample 0: label="{label}"  frames={len(frame_bytes)}')
print(f'Frame[0] size: {len(frame_bytes[0])} bytes')


## Cell 5 — Build DataLoaders

In [ ]:
from wita_v2.datasets.dataset import make_dataloaders

train_loader, val_loader = make_dataloaders(cfg, samples, converter)
print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')

# Sanity check one batch
clips, labels, input_lens, label_lens = next(iter(train_loader))
print(f'clips      : {clips.shape}  {clips.dtype}')    # [B, T, C, H, W]
print(f'labels     : {labels.shape} {labels.dtype}')   # [B, L]
print(f'input_lens : {input_lens.tolist()}')
print(f'label_lens : {label_lens.tolist()}')

# Decode a label to verify
gt = converter.decode(labels[0, :label_lens[0].item()].int(),
                       torch.IntTensor([label_lens[0].item()]))
print(f'GT word[0] : "{gt}"')


## Cell 6 — Build model

In [ ]:
from wita_v2.models.hybrid_model import build_model
# Note: prepare_attn_targets removed — CTC-only model; no attention targets needed

model = build_model(cfg).to(cfg.device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}  (frozen backbone excluded)')
print()
print('  Encoder  :', cfg.encoder.arch.upper(), '— out_dim', cfg.encoder.out_dim)
print('  Recurrent:', cfg.recurrent.arch.upper(),
      '— hidden', cfg.recurrent.hidden_size, '× 2 (bi)')
print('  CTC head  → vocab', cfg.vocab.ctc_vocab_size)
print('  (Attention decoder removed — CTC-only architecture)')

# Forward sanity check (no OOM)
clips_b    = clips.to(cfg.device)
labels_b   = labels.to(cfg.device)
il_b       = input_lens.to(cfg.device)
ll_b       = label_lens.to(cfg.device)

with torch.no_grad():
    ctc_log_probs, enc_lens = model(clips_b, il_b)
print(f'\nCTC log_probs : {ctc_log_probs.shape}   (B, T\', {cfg.vocab.ctc_vocab_size})')
print(f'enc_lens      : {enc_lens.tolist()}  (pass to CTCLoss as input_lengths)')

if torch.cuda.is_available():
    print(f'VRAM after forward: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print('Forward pass OK ✓')


## Cell 7 — Train

Features enabled:
- **Mixed precision** (AMP) — halves VRAM bandwidth on T4
- **Gradient accumulation** — effective batch = `batch_size × accum_steps = 8`
- **Backbone unfreezing** — backbone frozen for first `unfreeze_after_epoch` epochs, then end-to-end fine-tuning
- **Checkpointing** — `latest.pt` every epoch + `best.pt` on CER improvement
- **Resume** — set `cfg.train.resume_path` to continue after a crash
- **VAL_LIMIT** — fast 50-batch validation pass each epoch

In [ ]:
from wita_v2.training.trainer import train

best_model = train(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    converter    = converter,
    cfg          = cfg,
)
print('Training complete.')


## Cell 8 — Final evaluation (full val set, CTC decoder)

In [ ]:
from wita_v2.evaluation.evaluator import evaluate_cer, print_sample_table

print('Running full validation (no batch limit) …')

cer_ctc, _ = evaluate_cer(
    best_model, val_loader, converter, cfg,
    decode_mode='ctc', max_batches=None,
)
print(f'Final Val CER (CTC greedy) : {cer_ctc:.4f}')

print_sample_table(
    best_model, val_loader, converter, cfg,
    epoch=cfg.train.num_epochs, max_batches=None,
)

print()
print('─' * 50)
print('Reference baselines')
print('  Original WiTA (3D-ResNet + CTC)         : ~0.30')
print('  + Augmentation only                     : ~0.24–0.26')
print('  + Hybrid CTC+Attn only                  : ~0.22–0.25')
print('  VideoMAE + CTC target (Phase 1)         : ~0.15–0.20')
print(f'  This run (CTC greedy)                  : {cer_ctc:.4f}')


## Cell 9 — Export for Phase 2

In [ ]:
import os, torch

export_path = os.path.join(cfg.train.checkpoint_dir, 'phase1_export.pt')
torch.save({
    'model_state_dict':  best_model.state_dict(),
    'ctc_vocab_size':    cfg.vocab.ctc_vocab_size,
    'attn_vocab_size':   cfg.vocab.attn_vocab_size,
    'lang':              cfg.data.lang,
    'encoder_arch':      cfg.encoder.arch,
    'encoder_dim':       cfg.encoder.out_dim,
    'val_cer_ctc':       cer_ctc,
}, export_path)

sz = os.path.getsize(export_path) / 1e6
print(f'Model exported → {export_path}  ({sz:.1f} MB)')
print('Ready for Phase 2 (Video Swin backbone, CLIP re-ranking).')


## Cell 10 — Optional: swap config for a quick ablation

Re-run from Cell 6 with any of these config tweaks:

In [ ]:
# ── Ablation: no recurrent head (direct Linear on VideoMAE tokens) ────
# cfg.recurrent.arch = 'none'

# ── Ablation: GRU instead of LSTM ────────────────────────────────────
# cfg.recurrent.arch = 'gru'

# ── Ablation: Transformer recurrent head ─────────────────────────────
# cfg.recurrent.arch = 'transformer'

# ── Ablation: keep backbone frozen for all epochs ─────────────────────
# cfg.train.unfreeze_after_epoch = 999

# ── Ablation: unfreeze backbone immediately (epoch 0) ─────────────────
# cfg.train.unfreeze_after_epoch = 0

# ── Smoke test: fewer temporal tokens (16 frames → T'=8, faster) ─────
# NOTE: T'=8 is too short for many long English words after sep-token
# insertion (e.g. "suggestion" → 11 tokens). CTCLoss returns inf for
# those and zero_infinity=True silences them — use 16 frames ONLY for
# a quick smoke test, not for a real training run.
# cfg.encoder.videomae_num_frames = 16

# ── Ablation: no discriminative LR (single param group at head LR) ───
# cfg.train.backbone_lr_mult = 1.0

# ── Ablation: random-init VideoMAE (no pretrained weights) ────────────
# cfg.encoder.pretrained = False

# ── Ablation: no augmentation ─────────────────────────────────────────
# from wita_v2.configs.default import AugConfig
# cfg.aug = AugConfig(
#     mirror_prob=0, rotation_deg=0, brightness=0, contrast=0,
#     saturation=0, hue=0, grayscale_prob=0,
#     drop_frames_prob=0, temporal_crop_ratio=(1.0, 1.0),
# )

print('Edit this cell and re-run from Cell 6 to run ablations.')
